Task 1 - Ingest Bronze -> Task 2 - Update Silver -> Task 3. - Validate
This is a Directed Acyclic Graph(DAG): tasks are connected through dependencies, with no cycles. If Bronze ingestion fails, Silver should not run. If Silver fails, final validation should not run. That is primary purpose of orchestration

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

catalog = "workspace"
schema = "incremental_project"

base_path = f"/Volumes/{catalog}/{schema}/source_files"

source_path = f"{base_path}/incoming_clean"
schema_path = f"{base_path}/_schemas/orders_v3"
checkpoint_path = f"{base_path}/_checkpoints/orders_v3"

bronze_table = f"{catalog}.{schema}.orders_bronze_v3"

Concept is data contract. StructType defines the columns Auto Loader should expect

In [ ]:
order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_amount", StringType(), True),
    StructField("updated_at", StringType(), True),
    StructField("_rescued_data", StringType(), True)
])

spark.readStream is Structured Streaming, .format("cloudFiles") is Auto Loader, .schema(order_schema) is source schema, _metadata.file_path is data lineage, .writeStream is streaming output, .format("delta") is Delta Lake, checkpointLocation is progress tracking and recovery, availableNow=True incremental and finite processing run, and query.awaitTermination is wait until ingestion finishes

Job has to wait for completion because by waiting for ingestion operation to finish, the notebook task can complete before dependent Silver task begins

In [ ]:
assert spark.catalog.tableExists(bronze_table), (
    "Expected the existing Step 3 Bronze table"
)

orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(order_schema)
    .load(source_path)
)

bronze_stream = (
    orders_stream
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

if query.exception() is not None:
    raise RuntimeError(str(query.exception()))

print("Bronze ingestion completed")

We use >= 10 rather than exactly 10 because future batches should increase # of bronze records

In [ ]:
bronze_df = spark.table(bronze_table)

total = bronze_df.count()

print("Bronze records:", total)

assert total >= 10, (
    f"Unexpected Bronze row count: {total}"
)

print("Bronze task validation passed")